In [14]:
import numpy as np
from MultiHeadAttention import MultiHeadAttention, Layer_Dense, sigmoid, binary_cross_entropy, binary_cross_entropy_backward, Embedding

np.random.seed(0)

# -----------------------------
# 1. Tiny sentiment dataset
# -----------------------------

In [15]:
sentences = [
    # positive
    "the movie was good",
    "the film was great",
    "i love this movie",
    "the movie was not bad",
    "the film was very good",
    "the movie was amazing",
    # negative
    "the movie was bad",
    "the film was terrible",
    "i hate this movie",
    "the movie was not good",
    "the film was very bad",
    "the movie was boring"
]

labels = [
    1, 1, 1, 1, 1, 1,   # positive = 1
    0, 0, 0, 0, 0, 0    # negative = 0
]

### Build vocabulary from the sentences

In [16]:
def build_vocab(sentences, extra_tokens=None):
    if extra_tokens is None:
        extra_tokens = ["<pad>", "<unk>"]
    vocab = list(extra_tokens)
    for s in sentences:
        for w in s.strip().split():
            if w not in vocab:
                vocab.append(w)
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    return vocab, word2idx, idx2word

In [17]:
vocab, word2idx, idx2word = build_vocab(sentences)
V = len(vocab)
pad_idx = word2idx["<pad>"]
unk_idx = word2idx["<unk>"]

print("Vocab:", vocab)
print("Vocab size:", V)

Vocab: ['<pad>', '<unk>', 'the', 'movie', 'was', 'good', 'film', 'great', 'i', 'love', 'this', 'not', 'bad', 'very', 'amazing', 'terrible', 'hate', 'boring']
Vocab size: 18


### Convert sentences to sequences of token indices

In [18]:
def encode_sentence(s, word2idx, max_len):
    tokens = s.strip().split()
    ids = [word2idx.get(w, unk_idx) for w in tokens]
    # pad / truncate to max_len
    if len(ids) < max_len:
        ids = ids + [pad_idx] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return np.array(ids, dtype=np.int32)

In [19]:
max_len = max(len(s.split()) for s in sentences)  # no extra pad beyond max sentence length
T = max_len

X_ids = np.stack([encode_sentence(s, word2idx, max_len) for s in sentences])   # (N, T)
y = np.array(labels, dtype=np.float32).reshape(-1, 1)                           # (N, 1)
N = X_ids.shape[0]

### One-hot encode: we want time-major for the attention layer, so per sample we'll do (T, V)

In [23]:
def one_hot(ids, V):
    # ids: (T,)
    x = np.zeros((ids.shape[0], V), dtype=np.float32)
    for t, idx in enumerate(ids):
        x[t, idx] = 1.0
    return x

# -----------------------------
# 2. Model definition
# -----------------------------

In [8]:
d_model = 16 # embedding dimension (must be divisible by num_heads)
num_heads = 4
embedding = Embedding(V, d_model)
attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
dense = Layer_Dense(n_inputs=d_model, n_neurons=1)

### Helper: simple mean pooling over time dimension

In [9]:
def mean_pool(sequence):
    """
    sequence: (T, d_model)
    returns: (1, d_model) mean over T
    """
    return np.mean(sequence, axis=0, keepdims=True)  # (1, d_model)

# -----------------------------
# 3. Training loop (SGD)
# -----------------------------

In [10]:
learning_rate = 0.5
n_epochs = 200

In [11]:
for epoch in range(n_epochs):
    epoch_loss = 0.0

    # Shuffle samples each epoch
    indices = np.arange(N)
    np.random.shuffle(indices)

    for idx in indices:
        # ---- Forward pass for one sample ----
        ids = X_ids[idx]                # (T,)
        label = y[idx:idx+1]            # (1, 1)

        # One-hot input, time-major for attention
        x_onehot = one_hot(ids, V)         # (T, V)
        x_embed  = embedding.forward(x_onehot)  # (T, d_model)

        # Self-attention
        attn_out, alpha = attn.forward(x_embed)   # (T, d_model), (num_heads, T, T)

        # Pool across time -> sentence representation
        pooled = mean_pool(attn_out)       # (1, d_model)

        # Dense -> logit
        logit = dense.forward(pooled)      # (1, 1)

        # Sigmoid -> probability
        pred = sigmoid(logit)              # (1, 1)

        # Loss
        loss = binary_cross_entropy(pred, label)
        epoch_loss += loss

        # ---- Backward pass ----
        # dL/d(pred)
        d_pred = binary_cross_entropy_backward(pred, label)   # (1, 1)

        # pred = sigmoid(logit) => dL/dlogit = dL/dpred * pred * (1 - pred)
        d_logit = d_pred * pred * (1.0 - pred)                # (1, 1)

        # Through dense layer
        d_pooled = dense.backward(d_logit)                    # (1, d_model)

        # Through mean pooling:
        # pooled = (1/T) * sum_t attn_out[t]
        # So each time step gets d_attn_out[t] = d_pooled / T
        d_attn_out = np.repeat(d_pooled / T, T, axis=0)       # (T, d_model)

        # Through attention
        d_x_embed = attn.backward(d_attn_out)
        embedding.backward(d_x_embed)                     # (T, d_model), ignored for now

        # ---- SGD parameter update ----
        embedding.W -= learning_rate * embedding.dW
        attn.W_q -= learning_rate * attn.dW_q
        attn.W_k -= learning_rate * attn.dW_k
        attn.W_v -= learning_rate * attn.dW_v
        attn.W_o -= learning_rate * attn.dW_o
        dense.weights -= learning_rate * dense.dweights
        dense.biases  -= learning_rate * dense.dbiases

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{n_epochs} - avg loss: {epoch_loss / N:.4f}")


Epoch 1/200 - avg loss: 0.7443
Epoch 20/200 - avg loss: 0.7498
Epoch 40/200 - avg loss: 0.7098
Epoch 60/200 - avg loss: 0.5247
Epoch 80/200 - avg loss: 0.1044
Epoch 100/200 - avg loss: 0.0002
Epoch 120/200 - avg loss: 0.0001
Epoch 140/200 - avg loss: 0.0001
Epoch 160/200 - avg loss: 0.0001
Epoch 180/200 - avg loss: 0.0000
Epoch 200/200 - avg loss: 0.0000


# -----------------------------
# 4. Inspect attention on examples
# -----------------------------

In [12]:
def predict_and_show_attention(sentence):
    ids = encode_sentence(sentence, word2idx, max_len)
    x_onehot  = one_hot(ids, V)
    x_embed  = embedding.forward(x_onehot)
    attn_out, alpha = attn.forward(x_embed)

    pooled = mean_pool(attn_out)
    logit = dense.forward(pooled)
    prob = sigmoid(logit)[0, 0]
    pred_label = 1 if prob >= 0.5 else 0

    tokens = sentence.split()
    T_used = len(tokens)

    print("\nSentence:", sentence)
    print("Tokens:", tokens)
    print(f"Prediction: {'positive' if pred_label==1 else 'negative'} (p={prob:.3f})")

    print("\nAttention heads:")
    with np.printoptions(precision=2, suppress=True):
        for h in range(num_heads):
            print(f"\nHead {h}:")
            print(alpha[h][:T_used, :T_used])

### Test on seen and slightly modified examples

In [13]:
test_sentences = [
    "the movie was good",
    "the movie was bad",
    "the movie was not good",
    "the movie was not bad",
    "i love this movie",
    "i hate this movie",
]

for s in test_sentences:
    predict_and_show_attention(s)


Sentence: the movie was good
Tokens: ['the', 'movie', 'was', 'good']
Prediction: positive (p=1.000)

Attention heads:

Head 0:
[[0.19 0.22 0.18 0.21]
 [0.2  0.19 0.19 0.23]
 [0.19 0.21 0.18 0.22]
 [0.2  0.2  0.2  0.18]]

Head 1:
[[0.2  0.23 0.19 0.16]
 [0.17 0.38 0.17 0.08]
 [0.2  0.23 0.2  0.16]
 [0.2  0.15 0.2  0.25]]

Head 2:
[[0.19 0.21 0.19 0.21]
 [0.16 0.31 0.14 0.18]
 [0.19 0.2  0.18 0.22]
 [0.21 0.16 0.22 0.22]]

Head 3:
[[0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]

Sentence: the movie was bad
Tokens: ['the', 'movie', 'was', 'bad']
Prediction: negative (p=0.000)

Attention heads:

Head 0:
[[0.19 0.22 0.18 0.21]
 [0.2  0.19 0.2  0.21]
 [0.19 0.22 0.18 0.21]
 [0.2  0.21 0.2  0.2 ]]

Head 1:
[[0.19 0.22 0.18 0.2 ]
 [0.15 0.34 0.15 0.18]
 [0.19 0.22 0.19 0.2 ]
 [0.18 0.24 0.18 0.2 ]]

Head 2:
[[0.19 0.21 0.19 0.2 ]
 [0.15 0.3  0.14 0.2 ]
 [0.19 0.21 0.19 0.2 ]
 [0.18 0.24 0.18 0.2 ]]

Head 3:
[[0.38 0.   0.41 0.02]
 [0.42 0.   0.49 0.  ]
 [0.39 0.   0.42 0.01]
 [0.